In [2]:
import os
from openpyxl import load_workbook

# ====== 这里改成你的文件夹路径 ======
TARGET_DIR = r"/Users/bangongyi/Downloads/上传数据最新30省 - 测试 2"

# 要检查的列名
TARGET_COLUMN = "专业备注"

# 关键词
KEYWORDS = ["非免费师范"]


def find_excel_files_with_keywords(folder_path):
    matched_files = {}

    for root, _, files in os.walk(folder_path):
        for file_name in files:
            # 跳过 Excel 临时文件
            if file_name.startswith(("~$", ".~")):
                continue

            if not file_name.lower().endswith((".xlsx", ".xlsm")):
                continue

            file_path = os.path.join(root, file_name)

            try:
                wb = load_workbook(file_path, read_only=True, data_only=True)
                ws = wb.active

                # 找表头
                header_map = {}
                for col_idx, cell in enumerate(ws[1], start=1):
                    if cell.value is not None:
                        header_map[str(cell.value).strip()] = col_idx

                if TARGET_COLUMN not in header_map:
                    wb.close()
                    continue

                remark_col = header_map[TARGET_COLUMN]
                hit_keywords = set()

                for row in ws.iter_rows(min_row=2, values_only=True):
                    cell_value = row[remark_col - 1]
                    if cell_value is None:
                        continue

                    text = str(cell_value)

                    for keyword in KEYWORDS:
                        if keyword in text:
                            hit_keywords.add(keyword)

                    # 三个词都找到了就没必要继续扫了
                    if len(hit_keywords) == len(KEYWORDS):
                        break

                wb.close()

                if hit_keywords:
                    matched_files[file_name] = sorted(hit_keywords, key=KEYWORDS.index)

            except Exception as e:
                print(f"处理失败: {file_name}，错误: {e}")

    return matched_files


if __name__ == "__main__":
    result = find_excel_files_with_keywords(TARGET_DIR)

    print("\n=== 命中的 Excel 文件及关键词 ===")
    for file_name, keywords in result.items():
        print(f"{file_name} -> {', '.join(keywords)}")


=== 命中的 Excel 文件及关键词 ===
